# LBM Validation Analysis Notebook

**Purpose:** Analyze validation test results to prove stability fixes are treating root cause (治本) not symptoms (治标).

**Analysis Components:**
1. Grid convergence order computation and visualization
2. Peclet number sweep comparison
3. Energy conservation time-series analysis
4. Literature benchmark comparison plots
5. Flux limiter impact assessment

**Author:** LBM-CUDA Architecture Team  
**Date:** 2025-01-19

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os
from pathlib import Path

# Configure matplotlib for publication-quality plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Base directory for validation results
BASE_DIR = Path('/home/yzk/LBMProject/validation_results')
OUTPUT_DIR = Path('/home/yzk/LBMProject/docs/validation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Validation Analysis Environment Ready")
print(f"Results directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Grid Convergence Analysis

Demonstrates numerical convergence order to prove solution quality.

In [ ]:
def analyze_grid_convergence():
    """
    Load grid convergence data and compute convergence order.
    
    Returns:
        dict: Convergence metrics (order, errors, etc.)
    """
    convergence_dir = BASE_DIR / 'grid_convergence'
    
    # Load diagnostics from each resolution
    resolutions = ['coarse', 'medium', 'fine']
    dx_values = [4e-6, 2e-6, 1e-6]  # meters
    
    results = {'dx': [], 'T_max': [], 'V_max': []}
    
    for res, dx in zip(resolutions, dx_values):
        diag_file = convergence_dir / res / 'diagnostics.csv'
        if diag_file.exists():
            df = pd.read_csv(diag_file)
            results['dx'].append(dx * 1e6)  # Convert to microns
            results['T_max'].append(df['T_max'].iloc[-1])
            results['V_max'].append(df['V_max'].iloc[-1])
        else:
            print(f"Warning: {diag_file} not found")
    
    # Compute convergence order
    if len(results['dx']) == 3:
        # Temperature convergence
        T_error_coarse = abs(results['T_max'][0] - results['T_max'][1])
        T_error_medium = abs(results['T_max'][1] - results['T_max'][2])
        T_ratio = T_error_coarse / (T_error_medium + 1e-10)
        T_order = np.log2(T_ratio)
        
        # Velocity convergence
        V_error_coarse = abs(results['V_max'][0] - results['V_max'][1])
        V_error_medium = abs(results['V_max'][1] - results['V_max'][2])
        V_ratio = V_error_coarse / (V_error_medium + 1e-10)
        V_order = np.log2(V_ratio)
        
        results['T_order'] = T_order
        results['V_order'] = V_order
        results['T_errors'] = [T_error_coarse, T_error_medium]
        results['V_errors'] = [V_error_coarse, V_error_medium]
    
    return results

# Execute analysis
conv_results = analyze_grid_convergence()

# Display results
print("\n=== Grid Convergence Results ===")
print(f"Temperature convergence order: {conv_results.get('T_order', 'N/A'):.3f}")
print(f"Velocity convergence order:    {conv_results.get('V_order', 'N/A'):.3f}")
print("\nTarget: Order >= 1.0 for first-order accurate method")

# Create convergence plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature convergence
axes[0].plot(conv_results['dx'], conv_results['T_max'], 'o-', markersize=10, label='T_max')
axes[0].set_xlabel('Grid spacing (μm)', fontsize=14)
axes[0].set_ylabel('Peak Temperature (K)', fontsize=14)
axes[0].set_title(f'Temperature Convergence (order = {conv_results.get("T_order", 0):.2f})', fontsize=14)
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=12)

# Velocity convergence
axes[1].plot(conv_results['dx'], conv_results['V_max'], 's-', markersize=10, color='orange', label='V_max')
axes[1].set_xlabel('Grid spacing (μm)', fontsize=14)
axes[1].set_ylabel('Peak Velocity (m/s)', fontsize=14)
axes[1].set_title(f'Velocity Convergence (order = {conv_results.get("V_order", 0):.2f})', fontsize=14)
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grid_convergence.png', dpi=300, bbox_inches='tight')
print(f"\nSaved: {OUTPUT_DIR / 'grid_convergence.png'}")
plt.show()

## 2. Peclet Number Sweep Analysis

Verifies stability across different advection-diffusion regimes.

In [ ]:
def analyze_peclet_sweep():
    """
    Load Peclet sweep data and compare stability/accuracy.
    """
    peclet_dir = BASE_DIR / 'peclet_sweep'
    
    test_cases = [
        ('diffusion_dominated', 1.7, 58e-6),
        ('balanced', 17, 5.8e-6),
        ('advection_dominated', 170, 0.58e-6)
    ]
    
    results = {'name': [], 'Pe': [], 'alpha': [], 'T_max': [], 'V_max': [], 'stable': []}
    
    for name, Pe, alpha in test_cases:
        diag_file = peclet_dir / name / 'diagnostics.csv'
        if diag_file.exists():
            df = pd.read_csv(diag_file)
            results['name'].append(name)
            results['Pe'].append(Pe)
            results['alpha'].append(alpha)
            results['T_max'].append(df['T_max'].iloc[-1])
            results['V_max'].append(df['V_max'].iloc[-1])
            results['stable'].append(df['T_max'].iloc[-1] < 10000)
    
    # Create visualization
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig)
    
    # Plot 1: Temperature vs Pe
    ax1 = fig.add_subplot(gs[0, 0])
    colors = ['green' if s else 'red' for s in results['stable']]
    ax1.scatter(results['Pe'], results['T_max'], c=colors, s=200, alpha=0.7, edgecolors='black', linewidths=2)
    ax1.axhline(10000, color='red', linestyle='--', linewidth=2, label='Stability threshold')
    ax1.set_xlabel('Peclet Number', fontsize=14)
    ax1.set_ylabel('Peak Temperature (K)', fontsize=14)
    ax1.set_title('Stability Across Peclet Number Regimes', fontsize=14)
    ax1.set_xscale('log')
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Velocity vs Pe
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(results['Pe'], results['V_max'], c=colors, s=200, alpha=0.7, edgecolors='black', linewidths=2)
    ax2.set_xlabel('Peclet Number', fontsize=14)
    ax2.set_ylabel('Peak Velocity (m/s)', fontsize=14)
    ax2.set_title('Flow Velocity Across Regimes', fontsize=14)
    ax2.set_xscale('log')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Temperature time series for each case
    ax3 = fig.add_subplot(gs[1, :])
    for name, Pe in zip(results['name'], results['Pe']):
        diag_file = peclet_dir / name / 'diagnostics.csv'
        if diag_file.exists():
            df = pd.read_csv(diag_file)
            ax3.plot(df['time'] * 1e6, df['T_max'], label=f'Pe = {Pe:.1f}', linewidth=2)
    
    ax3.set_xlabel('Time (μs)', fontsize=14)
    ax3.set_ylabel('Peak Temperature (K)', fontsize=14)
    ax3.set_title('Temperature Evolution Across Peclet Regimes', fontsize=14)
    ax3.legend(fontsize=12)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'peclet_sweep_analysis.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {OUTPUT_DIR / 'peclet_sweep_analysis.png'}")
    plt.show()
    
    return results

# Execute analysis
peclet_results = analyze_peclet_sweep()

print("\n=== Peclet Sweep Summary ===")
for i, name in enumerate(peclet_results['name']):
    status = "✓ STABLE" if peclet_results['stable'][i] else "✗ UNSTABLE"
    print(f"{name:25} Pe={peclet_results['Pe'][i]:6.1f}  T_max={peclet_results['T_max'][i]:7.1f}K  {status}")

## 3. Energy Conservation Analysis

Verifies thermodynamic consistency of the solution.

In [ ]:
def analyze_energy_conservation():
    """
    Analyze energy balance time series.
    """
    energy_dir = BASE_DIR / 'energy_conservation' / 'test_run'
    energy_file = energy_dir / 'energy_diagnostics.csv'
    
    if not energy_file.exists():
        print(f"Energy diagnostics file not found: {energy_file}")
        return None
    
    # Load energy data
    df = pd.read_csv(energy_file)
    
    # Create comprehensive energy plot
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(3, 2, figure=fig)
    
    # Plot 1: Energy error vs time
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df['time'] * 1e6, df['error_percent'], linewidth=2, color='crimson')
    ax1.axhline(5.0, color='orange', linestyle='--', linewidth=1.5, label='5% threshold')
    ax1.axhline(-5.0, color='orange', linestyle='--', linewidth=1.5)
    ax1.fill_between(df['time'] * 1e6, -5, 5, alpha=0.2, color='green')
    ax1.set_xlabel('Time (μs)', fontsize=14)
    ax1.set_ylabel('Energy Error (%)', fontsize=14)
    ax1.set_title('Energy Conservation Error vs Time', fontsize=14)
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Power terms vs time
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(df['time'] * 1e6, df['P_laser'], label='Laser Input', linewidth=2)
    ax2.plot(df['time'] * 1e6, df['P_evap'], label='Evaporation Loss', linewidth=2)
    ax2.plot(df['time'] * 1e6, df['P_rad'], label='Radiation Loss', linewidth=2)
    ax2.set_xlabel('Time (μs)', fontsize=14)
    ax2.set_ylabel('Power (W)', fontsize=14)
    ax2.set_title('Energy Balance Components', fontsize=14)
    ax2.legend(fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Energy partitioning (pie chart at final time)
    ax3 = fig.add_subplot(gs[2, 0])
    final_P_laser = df['P_laser'].iloc[-10:].mean()
    final_P_evap = df['P_evap'].iloc[-10:].mean()
    final_P_rad = df['P_rad'].iloc[-10:].mean()
    final_P_internal = final_P_laser - final_P_evap - final_P_rad
    
    sizes = [final_P_evap, final_P_rad, final_P_internal]
    labels = [f'Evaporation\n({100*final_P_evap/final_P_laser:.1f}%)',
              f'Radiation\n({100*final_P_rad/final_P_laser:.1f}%)',
              f'Internal Energy\n({100*final_P_internal/final_P_laser:.1f}%)']
    colors = ['#ff9999', '#66b3ff', '#99ff99']
    
    ax3.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
    ax3.set_title('Energy Partitioning (Quasi-Steady)', fontsize=14)
    
    # Plot 4: Error histogram
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.hist(df['error_percent'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax4.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
    ax4.set_xlabel('Energy Error (%)', fontsize=14)
    ax4.set_ylabel('Frequency', fontsize=14)
    ax4.set_title('Error Distribution', fontsize=14)
    ax4.legend(fontsize=12)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'energy_conservation.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {OUTPUT_DIR / 'energy_conservation.png'}")
    plt.show()
    
    # Compute statistics
    stats = {
        'avg_error': df['error_percent'].abs().mean(),
        'max_error': df['error_percent'].abs().max(),
        'std_error': df['error_percent'].std(),
        'drift': df['error_percent'].iloc[-10:].mean() - df['error_percent'].iloc[:10].mean()
    }
    
    return stats

# Execute analysis
energy_stats = analyze_energy_conservation()

if energy_stats:
    print("\n=== Energy Conservation Statistics ===")
    print(f"Average error:  {energy_stats['avg_error']:.3f}%")
    print(f"Maximum error:  {energy_stats['max_error']:.3f}%")
    print(f"Std deviation:  {energy_stats['std_error']:.3f}%")
    print(f"Error drift:    {energy_stats['drift']:.3f}% (should be ~ 0)")
    print("\nTarget: Average error < 5%, no systematic drift")

## 4. Literature Benchmark Comparison

Quantitative validation against Mohr et al. (2020).

In [ ]:
def compare_with_literature():
    """
    Compare simulation results with literature values.
    """
    benchmark_dir = BASE_DIR / 'literature_benchmark' / 'test_run'
    diag_file = benchmark_dir / 'diagnostics.csv'
    
    # Literature reference values (Mohr et al. 2020)
    lit_values = {
        'T_peak': (2400, 2800, 2600),  # (min, max, mean) in K
        'length': (150e-6, 300e-6, 225e-6),  # melt pool length in m
        'depth': (50e-6, 100e-6, 75e-6),  # melt pool depth in m
        'width': (100e-6, 200e-6, 150e-6),  # melt pool width in m
        'v_max': (0.5, 1.0, 0.75)  # peak velocity in m/s
    }
    
    if not diag_file.exists():
        print(f"Benchmark diagnostics not found: {diag_file}")
        return None
    
    df = pd.read_csv(diag_file)
    
    # Extract simulation values (time-averaged over quasi-steady state)
    sim_T_peak = df['T_max'].iloc[-10:].mean()
    sim_v_max = df['V_max'].iloc[-10:].mean()
    
    # Create comparison bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Temperature comparison
    ax1 = axes[0]
    x_pos = [0, 1]
    values = [lit_values['T_peak'][2], sim_T_peak]
    errors = [[lit_values['T_peak'][2] - lit_values['T_peak'][0]], 
              [lit_values['T_peak'][1] - lit_values['T_peak'][2]]]
    
    bars = ax1.bar(x_pos, values, color=['steelblue', 'coral'], alpha=0.7, edgecolor='black', linewidth=2)
    ax1.errorbar([0], [lit_values['T_peak'][2]], yerr=errors, fmt='none', color='black', 
                 capsize=10, capthick=2, linewidth=2)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(['Literature\n(Mohr 2020)', 'Simulation'], fontsize=12)
    ax1.set_ylabel('Peak Temperature (K)', fontsize=14)
    ax1.set_title('Temperature Comparison', fontsize=14)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add percentage difference text
    rel_error_T = 100 * (sim_T_peak - lit_values['T_peak'][2]) / lit_values['T_peak'][2]
    ax1.text(0.5, max(values) * 1.05, f'Difference: {rel_error_T:+.1f}%', 
             ha='center', fontsize=12, fontweight='bold')
    
    # Velocity comparison
    ax2 = axes[1]
    values = [lit_values['v_max'][2], sim_v_max]
    errors_v = [[lit_values['v_max'][2] - lit_values['v_max'][0]], 
                [lit_values['v_max'][1] - lit_values['v_max'][2]]]
    
    bars = ax2.bar(x_pos, values, color=['steelblue', 'coral'], alpha=0.7, edgecolor='black', linewidth=2)
    ax2.errorbar([0], [lit_values['v_max'][2]], yerr=errors_v, fmt='none', color='black',
                 capsize=10, capthick=2, linewidth=2)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['Literature\n(Mohr 2020)', 'Simulation'], fontsize=12)
    ax2.set_ylabel('Peak Velocity (m/s)', fontsize=14)
    ax2.set_title('Velocity Comparison', fontsize=14)
    ax2.grid(True, alpha=0.3, axis='y')
    
    rel_error_v = 100 * (sim_v_max - lit_values['v_max'][2]) / lit_values['v_max'][2]
    ax2.text(0.5, max(values) * 1.05, f'Difference: {rel_error_v:+.1f}%',
             ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'literature_comparison.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {OUTPUT_DIR / 'literature_comparison.png'}")
    plt.show()
    
    return {
        'sim_T_peak': sim_T_peak,
        'sim_v_max': sim_v_max,
        'rel_error_T': rel_error_T,
        'rel_error_v': rel_error_v
    }

# Execute comparison
lit_comparison = compare_with_literature()

if lit_comparison:
    print("\n=== Literature Benchmark Results ===")
    print(f"Temperature: {lit_comparison['sim_T_peak']:.1f} K ({lit_comparison['rel_error_T']:+.1f}%)")
    print(f"Velocity:    {lit_comparison['sim_v_max']:.3f} m/s ({lit_comparison['rel_error_v']:+.1f}%)")
    print("\nTarget: Within ±20% for temperature, ±30% for geometry")

## 5. Flux Limiter Impact Assessment

Quantifies accuracy vs. efficiency trade-off.

In [ ]:
def analyze_flux_limiter_impact():
    """
    Compare simulations with and without flux limiter.
    """
    limiter_dir = BASE_DIR / 'flux_limiter_impact'
    
    cases = {
        'With Limiter': limiter_dir / 'with_flux_limiter',
        'Without Limiter': limiter_dir / 'without_flux_limiter'
    }
    
    results = {}
    
    for name, case_dir in cases.items():
        diag_file = case_dir / 'diagnostics.csv'
        if diag_file.exists():
            df = pd.read_csv(diag_file)
            results[name] = {
                'df': df,
                'T_max_final': df['T_max'].iloc[-1],
                'V_max_final': df['V_max'].iloc[-1],
                'runtime': float(open(case_dir / 'runtime.txt').read())
            }
    
    if len(results) != 2:
        print("Flux limiter impact data incomplete")
        return None
    
    # Create comparison plots
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig)
    
    # Plot 1: Temperature time series comparison
    ax1 = fig.add_subplot(gs[0, :])
    for name, data in results.items():
        ax1.plot(data['df']['time'] * 1e6, data['df']['T_max'], 
                label=name, linewidth=2.5, alpha=0.8)
    ax1.set_xlabel('Time (μs)', fontsize=14)
    ax1.set_ylabel('Peak Temperature (K)', fontsize=14)
    ax1.set_title('Temperature Evolution: With vs Without Flux Limiter', fontsize=14)
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Final value comparison
    ax2 = fig.add_subplot(gs[1, 0])
    x_pos = [0, 1]
    T_values = [results['With Limiter']['T_max_final'], 
                results['Without Limiter']['T_max_final']]
    
    ax2.bar(x_pos, T_values, color=['green', 'blue'], alpha=0.7, edgecolor='black', linewidth=2)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['With Limiter', 'Without Limiter'], fontsize=11)
    ax2.set_ylabel('Final Temperature (K)', fontsize=14)
    ax2.set_title('Solution Accuracy Comparison', fontsize=14)
    ax2.grid(True, alpha=0.3, axis='y')
    
    T_diff = 100 * abs(T_values[0] - T_values[1]) / T_values[1]
    ax2.text(0.5, max(T_values) * 1.05, f'Difference: {T_diff:.2f}%',
             ha='center', fontsize=12, fontweight='bold')
    
    # Plot 3: Runtime comparison
    ax3 = fig.add_subplot(gs[1, 1])
    runtime_values = [results['With Limiter']['runtime'],
                      results['Without Limiter']['runtime']]
    
    ax3.bar(x_pos, runtime_values, color=['green', 'blue'], alpha=0.7, edgecolor='black', linewidth=2)
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(['With Limiter', 'Without Limiter'], fontsize=11)
    ax3.set_ylabel('Runtime (s)', fontsize=14)
    ax3.set_title('Computational Efficiency', fontsize=14)
    ax3.grid(True, alpha=0.3, axis='y')
    
    speedup = runtime_values[1] / runtime_values[0]
    ax3.text(0.5, max(runtime_values) * 1.05, f'Speedup: {speedup:.2f}x',
             ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'flux_limiter_impact.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {OUTPUT_DIR / 'flux_limiter_impact.png'}")
    plt.show()
    
    return {
        'accuracy_diff': T_diff,
        'speedup': speedup
    }

# Execute analysis
limiter_results = analyze_flux_limiter_impact()

if limiter_results:
    print("\n=== Flux Limiter Impact ===")
    print(f"Accuracy difference: {limiter_results['accuracy_diff']:.2f}% (target: < 5%)")
    print(f"Speedup factor:      {limiter_results['speedup']:.2f}x (target: > 2x)")

## 6. Summary Report

Consolidated validation status.

In [ ]:
def generate_summary_report():
    """
    Generate comprehensive validation summary.
    """
    print("\n" + "="*80)
    print("VALIDATION FRAMEWORK SUMMARY REPORT")
    print("="*80)
    print("")
    print("Purpose: Validate that stability fixes are treating root cause (治本)")
    print("         not superficial symptoms (治标)")
    print("")
    print("-"*80)
    print("TEST RESULTS:")
    print("-"*80)
    
    # Collect results from all tests
    tests = [
        ("Grid Convergence", conv_results.get('T_order', 0) >= 0.8, 
         f"Order = {conv_results.get('T_order', 0):.2f}"),
        ("Peclet Number Sweep", all(peclet_results['stable']), 
         "All Pe regimes stable"),
        ("Energy Conservation", energy_stats and energy_stats['avg_error'] < 5.0,
         f"Error = {energy_stats['avg_error']:.2f}%" if energy_stats else "N/A"),
        ("Literature Benchmark", lit_comparison and abs(lit_comparison['rel_error_T']) < 20,
         f"ΔT = {lit_comparison['rel_error_T']:+.1f}%" if lit_comparison else "N/A"),
        ("Flux Limiter Impact", limiter_results and limiter_results['accuracy_diff'] < 5.0,
         f"Accuracy: {limiter_results['accuracy_diff']:.2f}%, Speedup: {limiter_results['speedup']:.2f}x" if limiter_results else "N/A")
    ]
    
    for test_name, passed, details in tests:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"{test_name:25} {status:10} {details}")
    
    print("")
    print("-"*80)
    print("OVERALL VERDICT:")
    print("-"*80)
    
    all_passed = all([t[1] for t in tests])
    
    if all_passed:
        print("")
        print("✓ ALL TESTS PASSED")
        print("")
        print("Conclusion: Stability fixes are treating ROOT CAUSE (治本)")
        print("")
        print("Evidence:")
        print("  - Solution converges with grid refinement (numerically sound)")
        print("  - Stable across all Peclet number regimes (robust)")
        print("  - Energy conserved (thermodynamically consistent)")
        print("  - Matches literature benchmarks (physically accurate)")
        print("  - Minimal accuracy loss from flux limiter (not over-dissipative)")
        print("")
        print("Recommendation: PROCEED with current fixes for production use")
    else:
        print("")
        print("⚠ SOME TESTS FAILED")
        print("")
        print("Conclusion: Fixes may be partially superficial (治标)")
        print("")
        print("Action Items:")
        for test_name, passed, _ in tests:
            if not passed:
                print(f"  - Address {test_name} failure")
        print("")
        print("Recommendation: Debug failed tests before production deployment")
    
    print("")
    print("="*80)
    
    # Save report to file
    report_file = OUTPUT_DIR / 'validation_summary.txt'
    with open(report_file, 'w') as f:
        f.write("VALIDATION FRAMEWORK SUMMARY\n")
        f.write("="*80 + "\n\n")
        for test_name, passed, details in tests:
            status = "PASS" if passed else "FAIL"
            f.write(f"{test_name:25} {status:10} {details}\n")
        f.write("\n")
        f.write("Overall: " + ("ALL PASSED" if all_passed else "SOME FAILED") + "\n")
    
    print(f"\nReport saved to: {report_file}")

# Generate final report
generate_summary_report()